# IMPORT

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# DEFINE FOLDERS

In [2]:
DATASET_INPUT_FOLDER = Path("correlated_data_csv")
DATASET_OUTPUT_FOLDER = Path("clean_data_csv")
DELETED_STATIONS_FOLDER = Path("deleted_stations")

folders = [DATASET_INPUT_FOLDER, DATASET_OUTPUT_FOLDER, DELETED_STATIONS_FOLDER]
for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

# LOAD DATA

In [3]:
df = pd.read_csv(f"{DATASET_INPUT_FOLDER}/correlated_data_2.csv")

# RENAME FEATURES(COLUMNS)

In [4]:
"""
name,stationcode,ebike,mechanical,latitude,longitude,duedate,
numbikesavailable,numdocksavailable,capacity,is_renting,is_installed,nom_arrondissement_communes,is_returning,code_insee_commune,
weather_time,weather_interval,weather_temperature_2m,weather_is_day,weather_wind_speed_10m,weather_wind_direction_10m,weather_weather_code,source_file
"""

old_feature_names = ["name",
                     "stationcode" ,
                     "numbikesavailable", 
                     "nom_arrondissement_communes",  
                     "code_insee_commune", 
                     "weather_time",
                     "weather_temperature_2m", 
                     "weather_is_day",
                     "weather_wind_speed_10m",
                     "weather_wind_direction_10m",
                     "weather_weather_code"]
new_feature_names = ["station_name", 
                     "station_code",
                     "total_bikes_now", 
                     "commune_name", 
                     "postal_code", 
                     "timestamp",
                     "temperature", 
                     "is_day",
                     "wind_speed",
                     "wind_direction",
                     "weather_code"]

feature_name_dict = dict(zip(old_feature_names, new_feature_names))
df = df.rename(columns=feature_name_dict)

# CHECK

In [5]:
print(df.isna().sum().sort_values(ascending=False))

weather_interval     60531
weather_code         60531
wind_direction       60531
wind_speed           60531
is_day               60531
temperature          60531
timestamp            60531
station_name         40511
latitude             40511
longitude            40511
capacity             40511
commune_name         40511
postal_code          40511
ebike                22478
mechanical           22478
station_code         22478
is_installed             0
is_returning             0
is_renting               0
numdocksavailable        0
total_bikes_now          0
duedate                  0
source_file              0
dtype: int64


# KEEPING THE USABLE COLUMNS

In [6]:
# Keep only useful columns
keep_cols = [
    "station_code",
    "latitude",
    "longitude",
    "capacity",
    "total_bikes_now",
    "timestamp",
    "temperature",
    "wind_speed",
    "wind_direction",
    "weather_code",
    "is_day",
]

df = df[keep_cols]

print("Initial shape:", df.shape)

Initial shape: (4907829, 11)


# WEATHER TIME HANDLING

In [7]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

# Drop rows without time
df = df.dropna(subset=["timestamp"])

# Time features
df["hour"] = df["timestamp"].dt.hour
df["dayofweek"] = df["timestamp"].dt.dayofweek
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

# Cyclical encoding
df["sin_hour"] = np.sin(2 * np.pi * df["hour"] / 24)
df["cos_hour"] = np.cos(2 * np.pi * df["hour"] / 24)

# DROP INVALID ROWS(DOESNT CONTAIN THE DATA FOR TOTAL_BIKES_NOW)

In [8]:
df = df.dropna(subset=["total_bikes_now"])

# FILL MISSING WEATHER VALUES

In [9]:
weather_cols = [
    "temperature",
    "wind_speed",
    "wind_direction",
    "weather_code",
    "is_day",
]

for col in weather_cols:
    df[col] = df.groupby("station_code")[col].transform(
        lambda x: x.fillna(x.mean())
    )


# DROP BROKEN WEATHER ROWS

In [10]:
df = df.dropna(subset=weather_cols)

# DETECT/REMOVE BROKEN STATIONS

In [11]:
MIN_ROWS = 200
MIN_VARIANCE = 1e-2

station_stats = (
    df.groupby("station_code")["total_bikes_now"]
    .agg(["count", "var"])
)

broken_stations = station_stats[
    (station_stats["count"] < MIN_ROWS) |
    (station_stats["var"] < MIN_VARIANCE)
]

valid_stations = station_stats.index.difference(broken_stations.index)


# SAVE THE BROKEN STATIONS IN A FILE

In [12]:
broken_stations.to_csv(f"{DELETED_STATIONS_FOLDER}/broken_stations.csv")
print(f"Broken stations saved: {len(broken_stations)}")


Broken stations saved: 28


# REMOVE THEM FROM THE DATABASE

In [13]:
df = df[df["station_code"].isin(valid_stations)]

print("After station filtering:", df.shape)
print("Remaining stations:", df['station_code'].nunique())

After station filtering: (4763007, 16)
Remaining stations: 1474


# CHECK

In [14]:
print(df.isna().sum().sort_values(ascending=False))

station_code       0
latitude           0
longitude          0
capacity           0
total_bikes_now    0
timestamp          0
temperature        0
wind_speed         0
wind_direction     0
weather_code       0
is_day             0
hour               0
dayofweek          0
is_weekend         0
sin_hour           0
cos_hour           0
dtype: int64


# SAVE THE FINAL CLEAN DATASET

In [15]:
df.to_csv(f"{DATASET_OUTPUT_FOLDER}/clean_bike_weather_data.csv", index=False)
print("Clean dataset saved: clean_bike_weather_data.csv")

Clean dataset saved: clean_bike_weather_data.csv
